# Organizing Machine Learning Datasets and Cross-Validation Splits

**Session:** 2026-10-31 · Competition tasks and metrics  
**Module:** 01 — Python Fundamentals & Algorithmic Thinking

High-performing machine learning systems depend on disciplined data organization. Without clean partitioning, leaks between training and validation sets artificially inflate performance estimates and lead to severe test-set degradation ("shakeup"). This notebook covers dataset structuring, image ingestion with OpenCV, color-space management, batch visual inspection, and leak-free stratified splitting.

## Learning Outcomes

- Structure project datasets cleanly across raw inputs, metadata manifests, and partition splits.
- Ingest images using OpenCV (`cv2`) and understand the BGR versus RGB color representation.
- Resize, normalize, and visualize image batches in Matplotlib grid layouts.
- Implement random Train / Validation / Test partitioning from first principles.
- Implement Stratified Splitting to preserve class balance across all evaluation folds.
- Prevent data leakage by isolating preprocessor transformations strictly to training partitions.


In [ ]:
import csv
from pathlib import Path

# Load and inspect dataset manifest
manifest_path = Path("../data/sample_manifest.csv")
records = []
with manifest_path.open("r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        records.append(row)

print(f"Total manifest records: {len(records)}")
for r in records:
    print(
        f"  ID: {r['image_id']} | File: {r['filename']:<24} | Class: {r['class_name']:<6} | Split: {r['split']}"
    )

## 1. Image Ingestion and Color Spaces (BGR vs RGB)

OpenCV loads images in **BGR** (Blue, Green, Red) channel order by default due to historical hardware standards. Matplotlib and standard image libraries expect **RGB** (Red, Green, Blue). Plotting raw OpenCV arrays without conversion leads to distorted, inverted color rendering.


In [ ]:
from pathlib import Path

import cv2

# Load sample image with OpenCV
sample_img_path = Path("../data/sample_images/cat_01.png")
bgr_image = cv2.imread(str(sample_img_path))
assert bgr_image is not None, f"Failed to load {sample_img_path}"

# Convert from BGR to standard RGB
rgb_image = cv2.cvtColor(bgr_image, cv2.COLOR_BGR2RGB)

print(f"Image shape: {bgr_image.shape} (Height x Width x Channels)")
print(f"Pixel dtype: {bgr_image.dtype} | Range: [{bgr_image.min()}, {bgr_image.max()}]")

## 2. Batch Loading, Resizing, and Grid Visualization

Computer vision models require fixed input dimensions. We load all images in the collection, resize them to uniform dimensions ($64 	imes 64$), normalize pixel values to $[0.0, 1.0]$, and display them on a unified grid.


In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

images_dir = Path("../data/sample_images")
image_paths = sorted(images_dir.glob("*.png"))
loaded_images = []
labels = []

target_size = (64, 64)

for p in image_paths:
    raw = cv2.imread(str(p))
    rgb = cv2.cvtColor(raw, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(rgb, target_size, interpolation=cv2.INTER_AREA)
    normalized = resized.astype(np.float32) / 255.0
    loaded_images.append(normalized)
    labels.append(p.stem)

# Visualize on a 2x3 grid
fig, axes = plt.subplots(2, 3, figsize=(8, 5))
for idx, ax in enumerate(axes.flat):
    if idx < len(loaded_images):
        ax.imshow(loaded_images[idx])
        ax.set_title(labels[idx], fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Train / Validation / Test Splitting from First Principles

In supervised machine learning, the dataset is divided into three distinct roles:
1. **Training Set:** Used by optimization algorithms to update model parameters $\theta$.
2. **Validation Set:** Used during development for hyperparameter tuning, model selection, and early stopping.
3. **Test Set:** Kept unseen until final evaluation to provide an unbiased estimate of generalization.


In [ ]:
from collections.abc import Sequence
from typing import TypeVar

import numpy as np

T = TypeVar("T")


def train_val_test_split(
    data: Sequence[T],
    train_ratio: float = 0.70,
    val_ratio: float = 0.15,
    seed: int = 42,
) -> tuple[list[T], list[T], list[T]]:
    """Partition data into train, val, and test partitions deterministically."""
    assert train_ratio + val_ratio < 1.0, "Sum of train and val ratios must be < 1.0"
    rng = np.random.default_rng(seed)
    indices = np.arange(len(data))
    rng.shuffle(indices)

    n_total = len(data)
    n_train = int(n_total * train_ratio)
    n_val = int(n_total * val_ratio)

    train_idx = indices[:n_train]
    val_idx = indices[n_train : n_train + n_val]
    test_idx = indices[n_train + n_val :]

    train_data = [data[i] for i in train_idx]
    val_data = [data[i] for i in val_idx]
    test_data = [data[i] for i in test_idx]

    return train_data, val_data, test_data


sample_pool = [f"sample_{i:02d}" for i in range(20)]
train_s, val_s, test_s = train_val_test_split(sample_pool, 0.6, 0.2, seed=42)
print(
    f"Total: {len(sample_pool)} | Train: {len(train_s)} | Val: {len(val_s)} | Test: {len(test_s)}"
)

## 4. Stratified Splitting for Imbalanced Classes

When class distributions are non-uniform (e.g. 90% dogs, 10% cats), naive random splitting may leave the validation or test fold with zero instances of the minority class. **Stratified Splitting** guarantees that each partition inherits the exact class proportions of the overall dataset.


In [ ]:
from collections import defaultdict
from collections.abc import Sequence
from typing import TypeVar

import numpy as np

T = TypeVar("T")


def stratified_split(
    items: Sequence[T],
    class_labels: Sequence[int],
    train_ratio: float = 0.70,
    seed: int = 42,
) -> tuple[list[T], list[T]]:
    """Stratified split ensuring identical class proportions across train and val."""
    rng = np.random.default_rng(seed)
    class_to_indices = defaultdict(list)
    for idx, label in enumerate(class_labels):
        class_to_indices[label].append(idx)

    train_indices = []
    val_indices = []

    for _label, idxs in class_to_indices.items():
        arr = np.array(idxs)
        rng.shuffle(arr)
        split_point = int(len(arr) * train_ratio)
        train_indices.extend(arr[:split_point])
        val_indices.extend(arr[split_point:])

    train_items = [items[i] for i in sorted(train_indices)]
    val_items = [items[i] for i in sorted(val_indices)]

    return train_items, val_items


# Test stratification with an imbalanced 80/20 label pool
labels_pool = [0] * 40 + [1] * 10
items_pool = [f"item_{i}" for i in range(50)]

train_it, val_it = stratified_split(items_pool, labels_pool, train_ratio=0.8, seed=42)
train_labels = [labels_pool[int(x.split("_")[1])] for x in train_it]
val_labels = [labels_pool[int(x.split("_")[1])] for x in val_it]

print(f"Overall Class 1 Ratio: {labels_pool.count(1) / len(labels_pool):.2%}")
print(f"Train   Class 1 Ratio: {train_labels.count(1) / len(train_labels):.2%}")
print(f"Val     Class 1 Ratio: {val_labels.count(1) / len(val_labels):.2%}")
assert train_labels.count(1) / len(train_labels) == val_labels.count(1) / len(val_labels)
print("Stratification verified: exact class proportions preserved!")